In [7]:
import sys
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.dynamicframe import DynamicFrame
from awsglue.job import Job
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
glueContext = GlueContext(SparkContext.getOrCreate())

# Environment-specific Data Catalog configuration
# This will be set after we determine the environment in the next cell
db_name = None
tbl_name = None

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# Method 1: Environment-aware configuration with try-catch approach
try:
    # Try to get job parameters from Glue
    args = getResolvedOptions(sys.argv, ['JOB_NAME', 'ENVIRONMENT'])
    job_name = args['JOB_NAME']
    environment = args.get('ENVIRONMENT', 'DEV').upper()
except:
    # Local development mode - use default values
    job_name = "local-glue-job"
    environment = "DEV"
    args = {'JOB_NAME': job_name, 'ENVIRONMENT': environment}

print(f"Job name: {job_name}")
print(f"Environment: {environment}")

In [ ]:
# Environment-specific database and table configuration
env_config = {
    "DEV": {
        "db_name": "payments_dev",
        "tbl_name": "medicare_dev",
        "s3_bucket": "my-glue-bucket-dev",
        "s3_output_path": "s3://my-glue-bucket-dev/output/medicare/",
        "checkpoint_location": "s3://my-glue-bucket-dev/checkpoints/"
    },
    "TEST": {
        "db_name": "payments_test", 
        "tbl_name": "medicare_test",
        "s3_bucket": "my-glue-bucket-test",
        "s3_output_path": "s3://my-glue-bucket-test/output/medicare/",
        "checkpoint_location": "s3://my-glue-bucket-test/checkpoints/"
    },
    "PROD": {
        "db_name": "payments_prod",
        "tbl_name": "medicare_prod", 
        "s3_bucket": "my-glue-bucket-prod",
        "s3_output_path": "s3://my-glue-bucket-prod/output/medicare/",
        "checkpoint_location": "s3://my-glue-bucket-prod/checkpoints/"
    }
}

# Set configuration based on environment
config = env_config.get(environment, env_config["DEV"])
db_name = config["db_name"]
tbl_name = config["tbl_name"]
s3_output_path = config["s3_output_path"]

print(f"Using configuration for {environment}:")
print(f"  Database: {db_name}")
print(f"  Table: {tbl_name}")
print(f"  S3 Output: {s3_output_path}")

# Alternative approaches for environment configuration
import os

# Method 2: Using environment variables (useful for containerized deployments)
env_from_var = os.getenv('GLUE_ENVIRONMENT', 'DEV').upper()
print(f"Environment from ENV VAR: {env_from_var}")

# Method 3: Using Job Name pattern to infer environment
def infer_env_from_job_name(job_name):
    """Infer environment from job name pattern"""
    job_lower = job_name.lower()
    if 'prod' in job_lower:
        return 'PROD'
    elif 'test' in job_lower or 'staging' in job_lower:
        return 'TEST'
    else:
        return 'DEV'

inferred_env = infer_env_from_job_name(job_name)
print(f"Inferred environment from job name '{job_name}': {inferred_env}")

# Method 4: Environment-specific validation
def validate_environment_config(env, config):
    """Validate that required configuration exists for the environment"""
    required_keys = ['db_name', 'tbl_name', 's3_output_path']
    
    for key in required_keys:
        if key not in config or not config[key]:
            raise ValueError(f"Missing required configuration '{key}' for environment '{env}'")
    
    # Environment-specific validations
    if env == 'PROD':
        # Additional validation for production
        if 'dev' in config['db_name'].lower():
            raise ValueError("Production job cannot use dev database")
    
    print(f"✓ Configuration validation passed for {env} environment")

validate_environment_config(environment, config)
